<a href="https://colab.research.google.com/github/ManjulaNaga/hybrid_search_and_retrival/blob/main/hybrid_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -qU qdrant-client fastembed datasets transformers



Step1: Add imports


In [3]:
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint,
)
from transformers import AutoTokenizer

import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding

print(f"FastEmbed version: {fastembed.__version__}")

FastEmbed version: 0.8.0


2. Load datasets


We'll use Amazon ESCI (Exact, Substitute, Complement, Irrelevant) — a real-world product search relevance dataset. Each query-product pair has a human-labeled relevance grade:

https://huggingface.co/datasets/tasksource/esci

In [4]:
dataset = load_dataset("tasksource/esci", split="train")

README.md:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00011-2d36455632bef8(…): reconstructing file:   0%|          |  0.00B /  115MB            

data/train-00000-of-00011-2d36455632bef8(…): downloading bytes:           |  0.00B            

data/train-00001-of-00011-18b81793a48399(…): reconstructing file:   0%|          |  0.00B /  120MB            

data/train-00001-of-00011-18b81793a48399(…): downloading bytes:           |  0.00B            

...

data/test-00002-of-00004-a81cff173329b48(…): downloading bytes:           |  0.00B            

data/test-00003-of-00004-22af4ca7fa1313b(…): reconstructing file:   0%|          |  0.00B /  175MB            

data/test-00003-of-00004-22af4ca7fa1313b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2027874 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/652490 [00:00<?, ? examples/s]

In [5]:
# We'll select the first 1000 examples for this demo
dataset = dataset.select(range(1000))
dataset = dataset.filter(lambda x: x["product_locale"] == "us")

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
print(f"Total query-product pairs: {len(dataset)}")
print(f"\nFeatures: {dataset.column_names}")

Total query-product pairs: 919

Features: ['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_text']


 3. Data Cleaning

Real data is messy! The same product can appear multiple times (paired with different queries). For our product catalog (the documents we search over), we need unique products.

In [7]:
source_df = dataset.to_pandas() # convert a dataset object into a pandas DataFrame


In [8]:
# Deduplicate products (same product can appear with different queries)
df = source_df.drop_duplicates(
    subset=["product_text", "product_title", "product_bullet_point", "product_brand"]
)

In [9]:
print(f"Unique products in catalog: {len(df)}")
print(f"Total search queries: {len(source_df)}")
print(f"\n→ On average, each product appears in {len(source_df) / len(df):.1f} query-product judgments")

Unique products in catalog: 182
Total search queries: 919

→ On average, each product appears in 5.0 query-product judgments


In [10]:
sample = df.iloc[0]
print(f"Product Title:  {sample['product_title']}")
print(f"Brand:          {sample['product_brand']}")
print(f"ESCI Label:     {sample['esci_label']}")
print(f"Query:          {sample['query']}")

Product Title:  Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan
Brand:          Panasonic
ESCI Label:     Irrelevant
Query:           revent 80 cfm


In [11]:
df.head()

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,product_title,product_description,product_bullet_point,product_brand,product_color,product_text
0,0,revent 80 cfm,0,B000MOO21W,us,Irrelevant,0,1,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,None,WhisperCeiling fans feature a totally enclosed...,Panasonic,White,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...
2,1,revent 80 cfm,0,B07X3Y6B1V,us,Exact,0,1,Homewerks 7141-80 Bathroom Fan Integrated LED ...,None,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,80 CFM,Homewerks 7141-80 Bathroom Fan Integrated LED ...
3,2,revent 80 cfm,0,B07WDM7MQQ,us,Exact,0,1,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,None,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,White,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...
4,3,revent 80 cfm,0,B07RH6Z8KW,us,Exact,0,1,Delta Electronics RAD80L BreezRadiance 80 CFM ...,This pre-owned or refurbished product has been...,Quiet operation at 1.5 sones\nBuilt-in thermos...,DELTA ELECTRONICS (AMERICAS) LTD.,White,Delta Electronics RAD80L BreezRadiance 80 CFM ...
5,4,revent 80 cfm,0,B07QJ7WYFQ,us,Exact,0,1,Panasonic FV-08VRE2 Ventilation Fan with Reces...,None,The design solution for Fan/light combinations...,Panasonic,White,Panasonic FV-08VRE2 Ventilation Fan with Reces...


4. Create the Document Text

We concatenate relevant fields into a single text per product. This is what we'll embed.

In [12]:
df["combined_text"] = (
    df["product_title"] + "\n" + df["product_text"] + "\n"+  df["product_bullet_point"]
)

/tmp/ipykernel_1402/4243155573.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["combined_text"] = (


In [13]:
 #Preview
print(df["combined_text"].iloc[0][:1000] + "...")
print(f"\nTotal documents to embed: {len(df)}")

Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan
Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceiling Mounted Fan
Panasonic
White
None
WhisperCeiling fans feature a totally enclosed condenser motor and a double-tapered, dolphin-shaped bladed blower wheel to quietly move air
Designed to give you continuous, trouble-free operation for many years thanks in part to its high-quality components and permanently lubricated motors which wear at a slower pace
Detachable adaptors, firmly secured duct ends, adjustable mounting brackets (up to 26-in), fan/motor units that detach easily from the housing and uncomplicated wiring all lend themselves to user-friendly installation
This Panasonic fan has a built-in damper to prevent backdraft, which helps to prevent outside air from coming through the fan
0.35 amp
WhisperCeiling fans feature a totally enclosed condenser motor and a double-tapered, dolphin-shaped bladed blower wheel to quietly move air
Designed to give you continuous, trouble-fr

**5. Generating embeddings **

Now we are creating 2 vector representations for each product:
1. **Sparse** (SPLADE)
2. **Dense**(BGE-Large)

In [14]:
supported_models= (
    pd.DataFrame(TextEmbedding.list_supported_models())
    .sort_values("size_in_GB")
    .drop(columns=["model_file","additional_files"])
    .reset_index(drop=True)
)
supported_models

,model,sources,description,license,size_in_GB,dim,tasks
0,BAAI/bge-small-en-v1.5,"{'hf': 'qdrant/bge-small-en-v1.5-onnx-q', 'url...","Text embeddings, Unimodal (text), English, 512...",mit,0.067,384,{}
1,BAAI/bge-small-zh-v1.5,"{'hf': 'Qdrant/bge-small-zh-v1.5', 'url': 'htt...","Text embeddings, Unimodal (text), Chinese, 512...",mit,0.090,512,{}
2,snowflake/snowflake-arctic-embed-xs,"{'hf': 'snowflake/snowflake-arctic-embed-xs', ...","Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.090,384,{}
3,sentence-transformers/all-MiniLM-L6-v2,"{'hf': 'qdrant/all-MiniLM-L6-v2-onnx', 'url': ...","Text embeddings, Unimodal (text), English, 256...",apache-2.0,0.090,384,{}
4,jinaai/jina-embeddings-v2-small-en,"{'hf': 'xenova/jina-embeddings-v2-small-en', '...","Text embeddings, Unimodal (text), English, 819...",apache-2.0,0.120,512,{}
5,snowflake/snowflake-arctic-embed-s,"{'hf': 'snowflake/snowflake-arctic-embed-s', '...","Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.130,384,{}
6,nomic-ai/nomic-embed-text-v1.5-Q,"{'hf': 'nomic-ai/nomic-embed-text-v1.5', 'url'...","Text embeddings, Multimodal (text, image), Eng...",apache-2.0,0.130,768,{}
7,BAAI/bge-small-en,"{'hf': 'Qdrant/bge-small-en', 'url': 'https://...","Text embeddings, Unimodal (text), English, 512...",mit,0.130,384,{}
8,BAAI/bge-base-en-v1.5,"{'hf': 'qdrant/bge-base-en-v1.5-onnx-q', 'url'...","Text embeddings, Unimodal (text), English, 512...",mit,0.210,768,{}
9,sentence-transformers/paraphrase-multilingual-...,{'hf': 'qdrant/paraphrase-multilingual-MiniLM-...,"Text embeddings, Unimodal (text), Multilingual...",apache-2.0,0.220,384,{}


In [15]:
supported_models_1= (
    pd.DataFrame(SparseTextEmbedding
                 .list_supported_models())
    .sort_values("size_in_GB")
    .drop(columns=["sources","additional_files"])
    .reset_index(drop=False)
)
supported_models_1

,index,model,model_file,description,license,size_in_GB,requires_idf,vocab_size
0,3,Qdrant/bm25,mock.file,BM25 as sparse embeddings meant to be used wit...,apache-2.0,0.010,True,0
1,2,Qdrant/bm42-all-minilm-l6-v2-attentions,model.onnx,"Light sparse embedding model, which assigns an...",apache-2.0,0.090,True,30522
2,4,Qdrant/minicoil-v1,onnx/model.onnx,"Sparse embedding model, that resolves semantic...",apache-2.0,0.090,True,19125
3,1,prithvida/Splade_PP_en_v1,model.onnx,Independent Implementation of SPLADE++ Model f...,apache-2.0,0.532,None,30522
4,0,prithivida/Splade_PP_en_v1,model.onnx,Independent Implementation of SPLADE++ Model f...,apache-2.0,0.532,None,30522


In [16]:
# Models we will use
sparse_model_name = "prithvida/Splade_PP_en_v1"
dense_model_name = "BAAI/bge-large-en-v1.5"

In [17]:
sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)
dense_model = TextEmbedding(model_name=dense_model_name, batch_size=32)
print("✅ Both models loaded!")

/tmp/ipykernel_1402/1548607753.py:1: DeprecationWarning: The right spelling is prithivida/Splade_PP_en_v1. Support of this name will be removed soon, please fix the model_name
  sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Both models loaded!


In [18]:
# Helper functions
def make_sparse_embedding(texts: list[str]) -> list[SparseEmbedding]:
    return list(sparse_model.embed(texts, batch_size=32))

def make_dense_embedding(texts: list[str]):
    return list(dense_model.embed(texts))

6.2 Deep Dive: What Does a Sparse Embedding Look Like?

Let's embed a single sentence and inspect the internals. This is the most important cell in this section — it shows the "term expansion" magic of SPLADE.

In [19]:
test_text = "InterviewKickstart is a great place to learn Applied AI!"

In [20]:
sparse_embedding = make_sparse_embedding([test_text])

In [21]:
print(f"Indices: {sparse_embedding[0].indices.shape}")
print(f"Values:  {sparse_embedding[0].values.shape}")


Indices: (59,)
Values:  (59,)


In [22]:
print(f"Non-zero entries: {len(sparse_embedding[0].indices)}")
print(f"Full vector size:  30,522")
print(f"Actual storage:    {sparse_embedding[0].indices.nbytes + sparse_embedding[0].values.nbytes} bytes")

Non-zero entries: 59
Full vector size:  30,522
Actual storage:    944 bytes


In [23]:
def get_tokens_and_weights(sparse_embedding, model_name) -> dict[str, float]:
    """Decode sparse embedding indices back to human-readable tokens."""
    # Find the HuggingFace tokenizer for this model
    tokenizer_source = None
    for model_info in SparseTextEmbedding.list_supported_models():
        if model_info["model"].lower() == model_name.lower():
            tokenizer_source = model_info["sources"]["hf"]
            break

    if tokenizer_source is None:
        raise ValueError(f"Model {model_name} not found in supported models.")

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    token_weight_dict = {}
    for i in range(len(sparse_embedding.indices)):
        token = tokenizer.decode([sparse_embedding.indices[i]])
        weight = sparse_embedding.values[i]
        token_weight_dict[token] = weight

    # Sort by weight (highest first)
    return dict(sorted(token_weight_dict.items(), key=lambda item: item[1], reverse=True))

In [24]:
tokens = get_tokens_and_weights(sparse_embedding[0], sparse_model_name)

print("🔍 SPLADE Token Weights for: '" + test_text + "'\n")
print(f"{'Token':<20} {'Weight':>8}  {'Source'}")
print("─" * 55)

# Words that actually appear in the text (approximately)
original_words = set(test_text.lower().replace("!", "").split())

for token, weight in tokens.items():
    # Check if token (or a form of it) appears in original text
    is_original = any(token.replace("##", "") in word for word in original_words)
    source = "← in original text" if is_original else "← EXPANDED (not in text!)"
    print(f"{token:<20} {weight:>8.4f}  {source}")

config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

🔍 SPLADE Token Weights for: 'InterviewKickstart is a great place to learn Applied AI!'

Token                  Weight  Source
───────────────────────────────────────────────────────
ai                     2.3687  ← in original text
##kic                  2.1373  ← in original text
##tar                  2.0897  ← in original text
interview              2.0781  ← in original text
interviews             1.5263  ← EXPANDED (not in text!)
applied                1.5144  ← in original text
learn                  1.4884  ← in original text
ui                     1.1442  ← EXPANDED (not in text!)
apply                  1.0972  ← EXPANDED (not in text!)
place                  0.9840  ← in original text
application            0.9656  ← EXPANDED (not in text!)
##k                    0.9628  ← in original text
places                 0.9450  ← EXPANDED (not in text!)
##ks                   0.9367  ← in original text
learning               0.8577  ← EXPANDED (not in text!)
##t                    0.8

### 6.3 Deep Dive: What Does a Dense Embedding Look Like?

In [25]:
dense_embedding=make_dense_embedding(test_text)

In [26]:
print(f" Dense vector shape: {dense_embedding[0].shape}")
print(f"Every dimension has a value (min: {dense_embedding[0].min():.4f}, max: {dense_embedding[0].max():.4f})")
print(f"\nFirst 10 values: {dense_embedding[0][:10].round(4)}")


 Dense vector shape: (1024,)
Every dimension has a value (min: -0.0960, max: 0.2310)

First 10 values: [ 0.0137  0.0775 -0.0276 -0.0091 -0.0088 -0.037  -0.02    0.0288  0.0031
  0.0072]


6.4 Embed the Full Catalog

Now let's generate both embedding types for all 176 products. FastEmbed uses data parallelism across CPU cores to speed this up.

In [27]:
product_texts = df["combined_text"].tolist()
print(f"Embedding {len(product_texts)} products...")

Embedding 182 products...


In [28]:
print("⏳ Generating sparse embeddings (SPLADE)...")
# Ensure all elements in product_texts are strings to avoid TypeError
product_texts_cleaned = ["" if pd.isna(text) else str(text) for text in product_texts]
df["sparse_embedding"] = make_sparse_embedding(product_texts_cleaned)
print("✅ Done!")

⏳ Generating sparse embeddings (SPLADE)...
✅ Done!


/tmp/ipykernel_1402/3027556164.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sparse_embedding"] = make_sparse_embedding(product_texts_cleaned)


In [1]:
%%time
print("⏳ Generating dense embeddings (BGE-Large)...")
df["dense_embedding"] = make_dense_embedding(product_texts_cleaned)
print("✅ Done!")

⏳ Generating dense embeddings (BGE-Large)...


NameError: name 'make_dense_embedding' is not defined

 Indexing with Qdrant

We now have 176 products, each with two vector representations. We need a way to quickly find the most similar vectors to a query. But each vector type uses a different search strategy:

Dense vectors → Approximate Nearest Neighbor (ANN) search using algorithms like HNSW (Hierarchical Navigable Small World). This trades a tiny bit of accuracy for massive speed gains in high-dimensional continuous spaces.
Sparse vectors → Inverted index lookup, similar to how traditional search engines work. Each vocabulary token maps to a list of documents that activate it — Qdrant walks the query's non-zero tokens and aggregates matching documents.
Qdrant is a vector database that:

Stores vectors + associated metadata ("payload")
Builds the right index type for each vector automatically
Supports both dense and sparse vectors in the same collection
We'll use in-memory mode for this demo. In production, you'd run Qdrant as a server (Docker or Cloud).

### 7.1 Create the Collection

In [ ]:
client = QdrantClient(":memory:")  # In-memory for demo; use url="http://localhost:6333" in production

collection_name = "esci"

client.create_collection(
    collection_name,
    # Dense vector config: 1024-dim with cosine similarity
    vectors_config={
        "text-dense": VectorParams(
            size=1024,             # Must match model output dimension
            distance=Distance.COSINE,  # Cosine similarity for dense vectors
        )
    },
    # Sparse vector config: no fixed size needed (it's dynamic)
    sparse_vectors_config={
        "text-sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,    # Keep in RAM for speed
            )
        )
    },
)

print(f"✅ Collection '{collection_name}' created with dense + sparse vector support")

### 7.2 Prepare & Upload Points

Each "point" in Qdrant consists of:
- **ID** — unique identifier
- **Vectors** — our dense + sparse embeddings (stored under named keys)
- **Payload** — metadata (product text, product ID, etc.)

In [ ]:
def make_points(df: pd.DataFrame) -> list[PointStruct]:
    """Convert DataFrame rows into Qdrant PointStruct objects."""
    sparse_vectors = df["sparse_embedding"].tolist()
    product_texts = df["combined_text"].tolist()
    dense_vectors = df["dense_embedding"].tolist()
    rows = df.to_dict(orient="records")

    points = []
    for idx, (text, sparse_vector, dense_vector) in enumerate(
        zip(product_texts, sparse_vectors, dense_vectors)
    ):
        # Convert sparse embedding to Qdrant's SparseVector format
        sparse_vector = SparseVector(
            indices=sparse_vector.indices.tolist(),
            values=sparse_vector.values.tolist()
        )

        point = PointStruct(
            id=idx,
            payload={
                "text": text,
                "product_id": rows[idx]["product_id"],
            },
            vector={
                "text-sparse": sparse_vector,   # Sparse vector under its named key
                "text-dense": dense_vector.tolist(),  # Dense vector under its named key
            },
        )
        points.append(point)
    return points


points = make_points(df)
print(f"Prepared {len(points)} points for indexing")

In [ ]:
# Upload to Qdrant
client.upsert(collection_name, points)
print(f"✅ {len(points)} points indexed in Qdrant!")

## **Part 8: Hybrid Search**

Here's where it all comes together. For a single query, we:
1. Generate **both** sparse and dense query vectors
2. Fire **two parallel searches** against Qdrant
3. Get **two separate ranked lists** of results

### 8.1 The Search Function

In [ ]:
def search(query_text: str, top_k: int = 10):
    """
    Perform hybrid search: run dense AND sparse search in parallel,
    return both result lists.
    """
    # Step 1: Embed the query with BOTH models
    query_sparse_vectors = make_sparse_embedding([query_text])
    query_dense_vector = make_dense_embedding([query_text])

    # Step 2: Fire two searches in a single batch request
    search_results = client.query_batch_points(
        collection_name=collection_name,
        requests=[
            # Search 1: Dense (semantic similarity)
            QueryRequest(
                query=query_dense_vector[0].tolist(),
                using="text-dense",
                limit=top_k,
                with_payload=True,
            ),
            # Search 2: Sparse (lexical matching)
            QueryRequest(
                query=SparseVector(
                    indices=query_sparse_vectors[0].indices.tolist(),
                    values=query_sparse_vectors[0].values.tolist(),
                ),
                using="text-sparse",
                limit=top_k,
                with_payload=True,
            ),
        ],
    )

    # Extract the point lists from QueryResponse objects
    return [search_results[0].points, search_results[1].points]

### 8.2 Run the Search

Let's search for **"revent 80 cfm"** — a specific product query where exact keyword matching matters.

In [ ]:
query_text = "revent 80 cfm"
search_results = search(query_text)

dense_results, sparse_results = search_results[0], search_results[1]

print(f"Query: '{query_text}'")
print(f"\n{'='*60}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(dense_results[:5]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}")

print(f"\n{'='*60}")
print(f"SPARSE (Lexical/SPLADE) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(sparse_results[:5]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}")

---
## **Part 9: Reciprocal Rank Fusion (RRF)**

The Problem

We have two ranked lists with incompatible scores:

Dense search returns cosine similarity scores (-1 to 1, though typically positive for trained embeddings)
Sparse search returns dot product scores (0 to ∞, since SPLADE weights are non-negative)
We cannot simply average these scores — they're on completely different scales!

The Solution: Reciprocal Rank Fusion

RRF ignores the raw scores entirely and only uses rank positions. The formula:

RRF\_score(𝑖𝑡𝑒𝑚)=∑𝑟∈rank\_lists1𝛼+𝑟𝑎𝑛𝑘𝑟(𝑖𝑡𝑒𝑚)

Where:

𝛼  = smoothing constant (typically 60)
𝑟𝑎𝑛𝑘𝑟(𝑖𝑡𝑒𝑚)  = position of the item in rank list  𝑟  (1-indexed)
If the item is missing from a rank list, it gets default_rank = 1000
Worked Example

Suppose item "X" appears at rank 1 in dense results and rank 3 in sparse results:

RRF(X) = 1/(60+1) + 1/(60+3) = 0.0164 + 0.0159 = 0.0323
Item "Y" appears at rank 2 in dense but is missing from sparse (gets rank 1000):

RRF(Y) = 1/(60+2) + 1/(60+1000) = 0.0161 + 0.0009 = 0.0170
→ X ranks higher because it appeared in both lists!

Why alpha = 60?

The constant alpha controls how much the rank position matters:

Small alpha → Top ranks get much higher scores (more aggressive)
Large alpha → Scores are more uniform across ranks (more conservative)
60 is the standard from the original RRF paper (Cormack et al., 2009) — it works well in practice

***RRF function defination**

In [ ]:
def rrf(rank_lists, alpha=60, default_rank=1000):
    """
    Reciprocal Rank Fusion (RRF).

    Takes multiple rank lists and produces a single fused ranking.
    Only considers RANK POSITION, not raw scores — making it safe to
    combine results from different scoring systems.

    Args:
        rank_lists: List of [(item_id, rank), ...] lists
        alpha: Smoothing constant (default=60, from Cormack et al. 2009)
        default_rank: Rank assigned to items not in a list (high = penalty)

    Returns:
        Sorted list of (item_id, rrf_score) tuples
    """
    all_items = set(item for rank_list in rank_lists for item, _ in rank_list)
    item_to_index = {item: idx for idx, item in enumerate(all_items)}

    # Matrix: rows = items, cols = rank lists, filled with default_rank
    rank_matrix = np.full((len(all_items), len(rank_lists)), default_rank)

    for list_idx, rank_list in enumerate(rank_lists):
        for item, rank in rank_list:
            rank_matrix[item_to_index[item], list_idx] = rank

    # RRF formula: sum of 1/(alpha + rank) across all lists
    rrf_scores = np.sum(1.0 / (alpha + rank_matrix), axis=1)

    sorted_indices = np.argsort(-rrf_scores)  # Descending
    sorted_items = [(list(item_to_index.keys())[idx], rrf_scores[idx]) for idx in sorted_indices]

    return sorted_items

In [ ]:
# Toy example to build intuition
rank_list1 = [("A", 1), ("B", 2), ("C", 3)]   # System 1 thinks A > B > C
rank_list2 = [("B", 1), ("C", 2), ("D", 3)]   # System 2 thinks B > C > D
rank_list3 = [("A", 2), ("D", 1), ("E", 3)]   # System 3 thinks D > A > E

fused = rrf([rank_list1, rank_list2, rank_list3])

print("Fused ranking (RRF):")
print(f"{'Item':<6} {'RRF Score':>10}  Explanation")
print("─" * 55)
for item, score in fused:
    # Count how many lists this item appears in
    appearances = sum(1 for rl in [rank_list1, rank_list2, rank_list3]
                      if any(i == item for i, _ in rl))
    print(f"{item:<6} {score:>10.6f}  (appears in {appearances}/3 lists)")

### 9.2 Apply RRF to Our Search Results

1.   List item
2.   List item


In [ ]:
def rank_list(search_result: list[ScoredPoint]):
    """Convert Qdrant search results to (id, rank) pairs."""
    return [(point.id, rank + 1) for rank, point in enumerate(search_result)]


# Convert both result lists to rank lists
dense_rank_list = rank_list(search_results[0])
sparse_rank_list = rank_list(search_results[1])

print("Dense ranks:", dense_rank_list[:5], "...")
print("Sparse ranks:", sparse_rank_list[:5], "...")

In [ ]:
# Fuse the rankings!
rrf_rank_list = rrf([dense_rank_list, sparse_rank_list])

print(f"\n🏆 HYBRID SEARCH RESULTS for: '{query_text}'")
print(f"{'='*70}")

# Retrieve full records for the fused results
fused_records = client.retrieve(
    collection_name=collection_name,
    ids=[item[0] for item in rrf_rank_list]
)

# Create a lookup by ID
record_by_id = {r.id: r for r in fused_records}

for rank, (item_id, score) in enumerate(rrf_rank_list, 1):
    record = record_by_id[item_id]
    title = record.payload['text'].split('\n')[0][:75]

    # Check if this item was in dense, sparse, or both
    in_dense = any(id == item_id for id, _ in dense_rank_list)
    in_sparse = any(id == item_id for id, _ in sparse_rank_list)
    source = "BOTH" if (in_dense and in_sparse) else ("Dense only" if in_dense else "Sparse only")

    print(f"  {rank:>2}. [{source:<12}] {title}")

---
## **Part 10: Evaluation — Did It Work?**

Remember those ESCI labels? Let's check the ground truth relevance for our hybrid search results.

A perfect search engine would return **all Exact** labels for this query.

In [ ]:
ids = [item[0] for item in rrf_rank_list]

print(f"Query: '{query_text}'")
print(f"Results: {len(rrf_rank_list)} products")
print(f"\n{'Rank':<6} {'ESCI Label':<14} {'Product Title'}")
print("─" * 70)

label_counts = {}
for rank, idx in enumerate(ids, 1):
    label = df.iloc[idx]["esci_label"]
    title = df.iloc[idx]["product_title"][:60]
    emoji = {"Exact": "✅", "Substitute": "🔄", "Complement": "➕", "Irrelevant": "❌"}.get(label, "?")
    print(f"  {rank:<4} {emoji} {label:<12} {title}")
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"\n{'='*70}")
print(f"📊 Summary:")
for label, count in sorted(label_counts.items()):
    print(f"   {label}: {count}/{len(ids)} ({count/len(ids)*100:.0f}%)")

---
## **Part 11: Agentic RAG Pipeline (with LangChain)**


Traditional RAG = retrieve → generate. Agentic RAG adds a reasoning loop:

User Query
    │
    ▼
🤖 Agent (LLM) — powered by LangChain's create_agent (ReAct loop)
    │
    ├──→ Reason: "What should I search for?"
    │
    ├──→ Act: hybrid_search("quiet bathroom fan 80 cfm")
    │         │
    │         ▼
    │    Observe: Retrieved Context
    │         │
    ├──→ Reason: "Is this enough? Should I search again with a different query?"
    │
    ├──→ Act: hybrid_search("ventilation fan with built-in light")  ← refinement
    │         │
    │         ▼
    │    Observe: More Context
    │
    └──→ Final Answer (grounded in retrieved products)
The agent decides what to search for, whether to refine, and when it has enough context — instead of blindly retrieving once.

In [ ]:
!pip install -qU langchain langchain-openai langgraph

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

### 11.3 Define the hybrid search tool



In [4]:
from langchain.tools import tool


@tool
def hybrid_search_tool(query: str, top_k: int = 5) -> str:
    """Search the product catalog using hybrid retrieval (dense semantic + sparse lexical),
    fused with Reciprocal Rank Fusion. This combines the best of keyword matching
    and semantic understanding.

    Use this tool whenever you need to find products. You can call it multiple times
    with different queries to refine results or explore different aspects.

    Args:
        query: The search query to run against the product catalog.
        top_k: Number of results to return (default 5).
    """
    results = search(query, top_k=top_k)
    dense_ranks = rank_list(results[0])
    sparse_ranks = rank_list(results[1])
    fused = rrf([dense_ranks, sparse_ranks])

    records = client.retrieve(
        collection_name=collection_name,
        ids=[item[0] for item in fused[:top_k]],
    )
    record_by_id = {r.id: r for r in records}

### 11.4 Create the agent

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are an intelligent product search assistant with access to an e-commerce product catalog indexed in Qdrant with hybrid (dense + sparse) embeddings.

## Your tool
You have one tool: `hybrid_search_tool` — it runs both semantic and lexical search, then fuses results with Reciprocal Rank Fusion.

## Your agentic workflow
1. **Analyze** the user's query — what are they really looking for?
2. **Search** with a well-crafted query using the hybrid search tool.
3. **Evaluate** — are the results relevant enough? If not, search again with a refined or decomposed query.
4. **Decompose** complex questions into multiple searches if needed.
5. **Synthesize** a final answer grounded in the retrieved products. Cite product names/titles.

## Rules
- ALWAYS search before answering — never make up products.
- You may call the tool MULTIPLE times with different queries to get better coverage.
- If no relevant products are found, say so honestly.
- Keep answers concise but informative.
"""
agent = create_agent(
    model = "gpt-4o",
    tools=[hybrid_search_tool],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agentic RAG agent created!")

*italicized text*### 11.5 Test Agent

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "I need a quiet bathroom exhaust fan around 80 CFM. What are my options?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            msg = data["messages"][-1]
            print(f"\n[{step}] {type(msg).__name__}")
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🔧 {tc['name']}({tc['args']})")
            if msg.content:
                print(f"  {msg.content[:500]}")

In [ ]:
# Exact brand + model (sparse shines here)
result = agent.invoke({"messages": [{"role": "user", "content": "Do you have Broan-NuTone fans?"}]})
print(result["messages"][-1].content)

In [ ]:
# Conceptual / synonym-heavy (dense shines here)
result = agent.invoke({"messages": [{"role": "user", "content": "I want something to reduce moisture and mold in my shower area"}]})

print(result["messages"][-1].content)

In [ ]:
# Mixed — brand + concept (hybrid wins)
result = agent.invoke({"messages": [{"role": "user", "content": "Homewerks fan with a built-in LED light"}]})
print(result["messages"][-1].content)

In [ ]:
# Technical spec query (sparse catches numbers, dense catches intent)
result = agent.invoke({"messages": [{"role": "user", "content": "What's the most energy efficient exhaust fan under 1.0 sones?"}]})
print(result["messages"][-1].content)